In [0]:
%run /Workspace/F1_project/app_lib/0.ALDS_access_setup

In [0]:
%run /Workspace/F1_project/app_lib/0.1.configs_lib

In [0]:
dbutils.widgets.text("p_json_filepath", "")
dbutils.widgets.text("p_json_filename", "")
dbutils.widgets.text("p_json_schema", "")
dbutils.widgets.text("p_json_multiline", "")
dbutils.widgets.text("p_transformations_file", "")
dbutils.widgets.text("p_partitions", "")

v_json_filepath = dbutils.widgets.get("p_json_filepath")
v_json_filename = dbutils.widgets.get("p_json_filename")
v_json_schema = dbutils.widgets.get("p_json_schema")
v_json_multiline = dbutils.widgets.get("p_json_multiline").lower() == "true"
v_tx = dbutils.widgets.get("p_transformations_file")
v_transformations_file = "/Workspace/F1_project/custom_transformation/" + v_tx + ".ipynb"
v_partitions = dbutils.widgets.get("p_partitions")




In [0]:
json_df = spark.read\
        .schema(v_json_schema) \
        .option("multiLine", v_json_multiline) \
        .json(f"{v_json_filepath}")
       

In [0]:
if v_tx != "":
    %run $v_transformations_file

In [0]:
if v_tx != "":
    json_df = transformation(json_df)
    print("Transformations applied")
else:
    print("No transformations specified")

In [0]:
json_df = populate_ingestion_time(json_df)

In [0]:

part_cols = v_partitions.split(",") if v_partitions else []

writer = json_df.write.mode("overwrite")

if part_cols:
    writer = writer.partitionBy(*part_cols)

writer.parquet(f"{processed_folder}/{v_json_filename}")

In [0]:
display(json_df)

In [0]:
display(spark.read.parquet(f"{processed_folder}/{v_json_filename}")) 

In [0]:
dbutils.notebook.exit(f"{v_json_filename} has been ingested into processed container")